<!-- VIDEO: T / 생성형 AI 개론 — 개념 설명 (슬라이드) -->
<!-- VIDEO: P / 첫 LLM 호출 실습 — JSON·멀티턴·한국어 시나리오 (본 노트북) -->

# 01. 생성형 AI 개론

> **학습 목표**
> 1. 생성형 AI와 LLM의 동작 원리(다음 토큰 예측)를 이해한다.
> 2. 2026년 4월 기준 주요 LLM 모델군의 특성을 비교하고, 사용 가능한 옵션을 파악한다.
> 3. OpenAI SDK(또는 호환 엔드포인트의 Gemini)로 첫 LLM 호출을 수행한다.
> 4. `temperature`, `max_tokens` 등 핵심 파라미터의 의미와 영향을 직접 확인한다.
> 5. 자유 텍스트 응답 대신 **JSON 구조화 출력**을 받는 패턴을 익힌다.
>
> **선수 지식**: Python 기초 문법(함수·딕셔너리), 가상환경 설정 경험.

---

본 챕터에서는 LangChain을 사용하지 않고 **OpenAI SDK 단독**으로 LLM의 동작을 직접 다뤄봅니다. 이후 02번 노트북부터 LangChain의 추상화가 어떤 부분에서 가치를 제공하는지를 비교 학습하기 위한 기초입니다.

> **참고**: OpenAI 결제 등록 없이도 본 노트북을 동일하게 진행할 수 있도록, Google Gemini 키를 사용하는 분기 코드를 환경 설정 셀에 포함했습니다. Gemini는 OpenAI 호환 엔드포인트를 제공하므로 `base_url`만 교체하면 동일한 코드로 호출됩니다.

---

## 1. 생성형 AI란?

**생성형 AI(Generative AI)** 는 학습 데이터의 분포를 모델링하여 텍스트, 이미지, 코드, 음성 등 **새로운 콘텐츠를 생성**하는 인공지능을 의미합니다. 그중 **LLM(Large Language Model, 대규모 언어 모델)** 은 대규모 텍스트 코퍼스로 학습된 다음 토큰 예측 기반 모델로, ChatGPT, Claude, Gemini 등이 대표적 사례입니다.

### 모델 분류

| 구분 | 설명 | 대표 예시 |
|---|---|---|
| 판별 모델(Discriminative) | 입력에 대한 **분류 또는 예측** 수행 | 스팸 분류, 이미지 인식 |
| 생성 모델(Generative) | 데이터 분포를 학습해 **새로운 샘플 생성** | GPT, DALL·E, Stable Diffusion |

### LLM의 핵심 원리: 다음 토큰 예측

LLM은 주어진 문맥 다음에 등장할 확률이 가장 높은 **토큰(token)** 을 반복적으로 예측합니다. 번역, 요약, 추론, 코드 생성 등 복잡한 능력은 이 단순한 예측 과정의 반복에서 **창발(emergent)** 적으로 나타납니다.

### 비유: 다음 단어 예측기로서의 LLM

추상적인 개념을 일상적인 비유로 풀어 보겠습니다.

```
입력: "오늘 점심으로 김치"
   ↓ (다음에 올 가장 자연스러운 단어는?)
출력 후보: "찌개"   ← 92%
          "볶음밥" ← 5%
          "전"    ← 2%
          ...
```

LLM은 매 시점마다 **현재까지의 문맥에서 다음 토큰(≈단어 조각)이 무엇일지** 확률 분포를 계산하고, 그중 하나를 선택합니다. 이 과정을 한 토큰씩 반복적으로 수행하면 문장, 문단, 코드, 번역, 요약이 순차적으로 생성됩니다.

> 이러한 단순한 예측 메커니즘만으로 "삼각형의 넓이를 구하는 파이썬 함수를 작성해 줘"와 같은 복합적 추론이 가능해지는 현상을 **창발(emergent ability)** 이라고 부릅니다. 현재 LLM 연구의 주요 관심사 중 하나입니다.

### 토큰(token)의 개념

영어 "I love AI"는 일반적으로 `["I", " love", " AI"]`로 분할되어 **3토큰**으로 처리됩니다.
한국어 "안녕하세요"는 토크나이저에 따라 4~7토큰이 소비되며, 동일 문장 기준으로 영어 대비 토큰 수가 더 많아 비용 측면에서 불리할 수 있습니다.

OpenAI 토크나이저로 직접 확인하려면 다음 링크를 참고하세요: <https://platform.openai.com/tokenizer>

---

### 할루시네이션(hallucination)

LLM이 사실과 다른 내용을 그럴듯하게 생성하는 현상을 **할루시네이션(hallucination)** 이라고 합니다. 다음 토큰 예측 메커니즘은 사실 검증을 수행하지 않고 학습된 분포에서 가장 자연스러운 토큰을 선택하기 때문에, 정확성이 보장되지 않는 정보가 출력될 수 있습니다.

이를 보완하기 위한 대표적인 기법이 외부 지식을 검색해 응답에 활용하는 **RAG(Retrieval-Augmented Generation, 검색 증강 생성)** 이며, 06번 노트북에서 상세히 다룹니다.

---

## 2. 환경 설정

본 노트북은 LangChain 없이 **OpenAI SDK**만 사용합니다. 아래 세 가지 시나리오 중 본인 환경에 맞는 옵션을 선택하세요.

| 시나리오 | 필요한 키 | 모델 | 비고 |
|---------|----------|------|------|
| **A. Gemini 2.5 Flash-Lite** (권장) | `GOOGLE_API_KEY` | `gemini-2.5-flash-lite` | 한국어 품질 양호 |
| **B. OpenAI GPT-4.1-mini** | `OPENAI_API_KEY` | `gpt-4.1-mini` | 별도 설정 필요 |
| **C. Ollama Gemma 4 E2B** | 없음 | `gemma4:e2b` | 로컬 실행 |

> Google Gemini는 **OpenAI 호환 엔드포인트**를 공식 제공합니다. 따라서 OpenAI SDK 코드를 유지한 채 `base_url`만 교체하면 동일한 인터페이스로 모델을 호출할 수 있습니다.

`.env` 파일이 준비되지 않은 경우 [SETUP.md](./SETUP.md) 4번 섹션을 참고해 키를 발급한 뒤 진행하세요.

In [2]:
# 환경 변수 로드 + 자동 분기 (Gemini, OpenAI, Ollama)
from dotenv import load_dotenv
import os

load_dotenv()

# 사용 가능한 키 자동 감지
HAS_GEMINI = bool(os.getenv("GOOGLE_API_KEY"))
HAS_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

USE_PROVIDER = "gemini" if HAS_GEMINI else "openai" if HAS_OPENAI else "ollama"
print(f"✅ 환경 변수 로드 완료. 사용 제공자: {USE_PROVIDER}")

✅ 환경 변수 로드 완료. 사용 제공자: ollama


## 3. 주요 LLM 모델 비교 (2026년 4월 기준)

### 사용 가능한 모델 (권장)

| 제공처 | 모델 | 사용 한도 | 한국어 | 특징 |
|---|---|---|---|---|
| **Google AI Studio** | `gemini-2.5-flash-lite` | 15 RPM / 1,000 RPD | 우수 | 권장 |
| **Google AI Studio** | `gemini-2.5-flash` | 10 RPM / 250 RPD | 우수 | 추론 성능 강화 모델 |
| **Groq** | `qwen/qwen3-32b` | 사용 한도 제공 | 우수 | 매우 빠른 추론 속도, 128K 컨텍스트 |
| **Groq** | `llama-3.3-70b-versatile` | 사용 한도 제공 | 양호 | Meta 오픈웨이트 모델 |
| **Ollama (로컬)** | `gemma4:e2b` | **무제한** (로컬 실행) | 양호 | 2026-04 출시, 4GB |

### 추가 모델 (별도 설정 필요)

| 제공사 | 모델 | 출시 | 특징 |
|---|---|---|---|
| OpenAI | `gpt-4.1-mini` / `gpt-4.1-nano` | 2025 | 가성비와 응답 속도의 균형 |
| OpenAI | `gpt-5` (reasoning) | 2025 | 고난도 추론 및 코드 작성 |
| Anthropic | `claude-sonnet-4-6` / `claude-opus-4-6` | 2026 | 1M 토큰 컨텍스트 윈도우 |
| Mistral | `mistral-large` / `mistral-small` | 2025 | 유럽 기반 모델 |

### 프로젝트 실행 안내

- Gemini 사용 시 Gemini 2.5 Flash-Lite 키 한 개로 01~06번 노트북을 모두 실행할 수 있습니다.
- 02번 이후로는 LangChain v1.0의 `init_chat_model()`을 사용하므로, **모델 식별자만 변경**하면 동일 코드로 다른 제공사의 모델을 호출할 수 있습니다.
- 본 노트북(01번)은 OpenAI SDK 인터페이스로 Gemini, OpenAI 또는 Ollama를 호출합니다. Ollama는 `http://localhost:11434/v1`의 OpenAI 호환 엔드포인트를 사용합니다.

## 4. 첫 LLM 호출

위 셀에서 감지한 제공자(Gemini, OpenAI 또는 Ollama)에 따라 클라이언트와 모델 식별자를 자동 분기하여 호출합니다.

### 제공자별 클라이언트 설정

`OpenAI` 클래스 하나로 세 제공자의 모델을 호출할 수 있습니다. Gemini와 Ollama가 OpenAI 호환 엔드포인트를 제공하기 때문이며, 변경되는 부분은 `api_key`, `base_url`, `model` 식별자뿐입니다.

In [3]:
from openai import OpenAI

# 사용 제공자에 따라 클라이언트와 모델명을 자동 설정
if USE_PROVIDER == "gemini":
    # ✅ 사용 가능: Google Gemini (OpenAI 호환 엔드포인트)
    client = OpenAI(
        api_key=os.getenv("GOOGLE_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    MODEL = "gemini-2.5-flash-lite"
elif USE_PROVIDER == "openai":
    client = OpenAI()  # OPENAI_API_KEY 자동 감지
    MODEL = "gpt-4.1-mini"
else:
    # Ollama의 OpenAI 호환 엔드포인트
    client = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)
    MODEL = "gemma4:e2b"

# 첫 호출
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 친절한 한국어 AI 어시스턴트입니다."},
        {"role": "user", "content": "LLM이 무엇인지 한 문장으로 설명해주세요."},
    ],
)

print(f"[{MODEL}]")
print(response.choices[0].message.content)

[gemma4:e2b]
LLM(Large Language Model)은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어를 이해하고 생성하며 복잡한 질문에 답할 수 있는 능력을 갖춘 인공지능 모델입니다.


### 메시지 역할(role)

- **system**: 모델의 페르소나와 행동 규칙을 정의합니다. (예: "당신은 금융 분야 전문가입니다.")
- **user**: 사용자의 질문 또는 지시 사항입니다.
- **assistant**: 이전 턴에서 모델이 생성한 응답입니다. 멀티턴 대화 이력을 유지할 때 사용합니다.

### 응답 객체 구조 확인

In [4]:
# 응답 전체 구조 확인
print("모델:", response.model)
print("완료 이유:", response.choices[0].finish_reason)
print("토큰 사용량:")
print(f"  - 입력(prompt): {response.usage.prompt_tokens}")
print(f"  - 출력(completion): {response.usage.completion_tokens}")
print(f"  - 합계(total): {response.usage.total_tokens}")

모델: gemma4:e2b
완료 이유: stop
토큰 사용량:
  - 입력(prompt): 44
  - 출력(completion): 326
  - 합계(total): 370


## 5. 핵심 파라미터

LLM 호출 결과의 품질과 일관성에 직접적인 영향을 주는 주요 파라미터를 살펴봅니다.

### 5.1 Temperature — 출력 분포의 평탄화 정도

`temperature`는 다음 토큰을 샘플링할 때 사용되는 **확률 분포의 평탄화 정도**를 조절하는 값입니다. 값이 낮을수록 가장 확률이 높은 토큰이 선택될 가능성이 커지고, 값이 높을수록 분포가 평탄해져 다양성이 증가합니다.

| 값 | 동작 특성 | 적합한 작업 |
|---|---|---|
| `0.0` | 거의 결정적(deterministic) 출력. 동일 입력에 대해 거의 동일한 응답 | 데이터 추출, 분류, JSON 생성 |
| `0.7` (기본값) | 자연스러운 다양성 확보 | 일반 대화, 요약, 설명 |
| `1.5 ~ 2.0` | 분포가 매우 평탄해져 창의적이지만 일관성 저하 | 아이디어 발산, 변형 생성 |

**작업 유형별 권장 값**

- 데이터 추출, JSON 구조화, 코드 생성, 분류: `0` 또는 `0.1`
- 마케팅 카피 발상, 스토리텔링: `0.7 ~ 1.0`
- 다수의 변형 후보 생성: `1.0 ~ 1.5`

In [5]:
# 동일 프롬프트를 temperature 를 바꿔가며 호출
prompt = "5글자로 된 창의적인 커피숍 이름 하나만 지어주세요. 설명 없이 이름만 출력하세요."

for t in [0.0, 0.7, 1.5]:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=t,
    )
    print(f"temperature={t}: {r.choices[0].message.content.strip()}")

temperature=0.0: 새벽의 빛깔
temperature=0.7: 향기로운 꿈
temperature=1.5: 달콤한 쉼표


### 5.2 Max tokens — 출력 길이 제한

`max_tokens`는 모델이 생성할 수 있는 출력 토큰 수의 상한입니다. 비용을 통제하거나 응답 길이를 제어할 때 사용합니다. 한도를 초과하면 응답이 중간에 잘릴 수 있으므로, `finish_reason` 값이 `length`인지 함께 확인하는 것이 좋습니다.

In [6]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "파이썬을 30단어 이내로 소개해주세요."}],
    max_tokens=60,
)
print(r.choices[0].message.content)
print(f"\n출력 토큰: {r.usage.completion_tokens}")



출력 토큰: 60


## 6. 구조화 출력 (JSON)

실무에서는 모델의 자유 텍스트 대신 **사전에 정의된 스키마에 맞춘 응답**이 필요한 경우가 많습니다. OpenAI 호환 API에서는 `response_format={"type": "json_object"}` 또는 JSON Schema 기반 옵션을 통해 구조화 출력을 강제할 수 있습니다.

> 02번 이후 다룰 LangChain v1.0에서는 `ChatModel.with_structured_output(Schema)` 또는 `create_agent(..., response_format=ToolStrategy(Schema))` 형태로 동일한 작업을 보다 간결하게 처리할 수 있습니다.

In [13]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "사용자가 입력한 문장에서 의도를 분류해 JSON으로 응답하세요. "
                       '필드: {"intent": "질문|요청|불만|칭찬", "sentiment": "긍정|중립|부정", "keywords": [최대 3개]}',
        },
        {
            "role": "user",
            "content": "이 제품 진짜 괜찮네요! 혹시 대용량 버전도 있나요?",
        },
    ],
    response_format={"type": "json_object"},
)

import json
data = json.loads(r.choices[0].message.content)
print(json.dumps(data, ensure_ascii=False, indent=2))

{
  "intent": "질문",
  "sentiment": "긍정",
  "keywords": [
    "제품",
    "대용량 버전",
    "괜찮네요"
  ]
}


## 7. 멀티턴 대화

이전 응답을 메시지 리스트에 누적하면 모델이 **대화 맥락을 유지**한 상태로 다음 응답을 생성할 수 있습니다. 본 노트북에서는 이 패턴을 직접 구현하지만, 이후 노트북에서는 LangChain의 `Memory`와 LangGraph의 `Checkpointer`로 발전된 형태를 다룹니다.

In [14]:
history = [
    {"role": "system", "content": "당신은 ABC 주식회사의 친절한 AI 어시스턴트입니다."},
]

def chat(user_msg: str) -> str:
    history.append({"role": "user", "content": user_msg})
    r = client.chat.completions.create(model=MODEL, messages=history)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

print("User:", "안녕! 내 이름은 지훈이야.")
print("Bot :", chat("안녕! 내 이름은 지훈이야."))

print("\nUser:", "내가 아까 뭐라고 했지?")
print("Bot :", chat("내가 아까 뭐라고 했지?"))

User: 안녕! 내 이름은 지훈이야.
Bot : 안녕하세요, 지훈님! 만나서 반갑습니다. 😊

저는 ABC 주식회사에서 지훈님의 궁금증을 해결해 드리고 다양한 정보를 제공해 드릴 친절한 AI 어시스턴트입니다.

오늘 어떤 도움을 드릴까요? 주식 관련 질문이 있으신가요, 아니면 다른 궁금한 점이 있으신가요? 편하게 말씀해주세요! ✨

User: 내가 아까 뭐라고 했지?
Bot : 지훈님께서 아까 저에게 말씀하신 내용은 "**안녕! 내 이름은 지훈이야.**" 였습니다. 😊


---

## 8. 한국어 비즈니스 시나리오 실습

앞서 학습한 내용을 한국어 실무 상황에 적용해 봅니다. 세 시나리오 모두 동일한 호출 패턴(`client.chat.completions.create(...)`)을 사용하며, **시스템 프롬프트와 파라미터 설정**의 차이만으로 서로 다른 작업을 수행할 수 있음을 확인합니다.

### 8.1 회의록 요약 및 액션 아이템 추출

In [9]:
meeting_log = """
[기획팀 주간 회의 — 4월 22일 오전 10시]
김부장: 1분기 매출이 목표 대비 87% 달성됐는데, 신규 고객 유입은 12% 줄었어요.
이매니저: 광고 채널을 검토해야 할 것 같습니다. 인스타그램 ROAS가 떨어지고 있어서요.
박사원: 다음 주까지 채널별 ROAS 분석 자료 만들어 보겠습니다.
김부장: 좋아요. 그리고 5월 신제품 런칭 일정은 변동 없죠?
이매니저: 네, 5월 14일 그대로 진행 예정입니다. 그전에 브랜드 페이지 개편이 끝나야 하고요.
"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "회의록을 분석해 JSON으로 응답하세요. "
                                       '필드: {"summary": "2~3줄", '
                                       '"action_items": [{"owner": str, "task": str, "due": str}], '
                                       '"risks": [str]}'},
        {"role": "user", "content": meeting_log},
    ],
    response_format={"type": "json_object"},
    temperature=0,  # 추출 작업 → 결정적
)

import json
print(json.dumps(json.loads(r.choices[0].message.content), ensure_ascii=False, indent=2))

{
  "summary": "1분기 매출은 목표 대비 87%를 달성했으나 신규 고객 유입이 감소하여 광고 채널 검토가 필요하다는 논의가 이루어졌다. 특히 인스타그램 ROAS 하락에 대한 분석을 진행하기로 했으며, 5월 신제품 런칭 일정(5월 14일)은 유지되나 브랜드 페이지 개편 완료 시점을 고려해야 한다.",
  "action_items": [
    {
      "owner": "박사원",
      "task": "채널별 ROAS 분석 자료 작성",
      "due": "다음 주까지"
    }
  ],
  "risks": [
    "인스타그램 광고 채널의 ROAS 하락 문제",
    "신제품 런칭 일정(5월 14일) 준수를 위한 브랜드 페이지 개편 완료 시점 관리 필요"
  ]
}


### 8.2 한식 레시피의 비건 변환

`temperature=0.5` — 정확성과 창의성의 중간 지점에서, 적절한 식재료 대체안을 자연스럽게 제안받기 위한 설정입니다.

In [10]:
recipe = "김치찌개: 돼지고기 200g, 김치 300g, 두부 1/2모, 대파 1대, 다진 마늘, 멸치육수 2컵"

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 한식·비건 요리 전문가입니다. "
                                       "주어진 한식 레시피를 비건(vegan) 버전으로 재구성해주세요. "
                                       "재료 대체 이유도 한 줄씩 덧붙이세요."},
        {"role": "user", "content": recipe},
    ],
    temperature=0.5,
)
print(r.choices[0].message.content)

## 비건 김치찌개 레시피

| 재료 | 양 | 대체 이유 |
| :--- | :--- | :--- |
| **식물성 고기 (대체)** | 200g | 돼지고기 대신 식물성 단백질을 사용하여 고기의 식감과 포만감을 대체합니다. |
| 김치 | 300g | 그대로 사용합니다. |
| 두부 | 1/2모 | 그대로 사용합니다. |
| 대파 | 1대 | 그대로 사용합니다. |
| 다진 마늘 | 적당량 | 그대로 사용합니다. |
| **채소 육수** | 2컵 | 멸치육수 대신 채소 육수를 사용하여 비건 기반의 감칠맛을 냅니다. |
| **감칠맛 추가 (선택)** | 1 작은술 | 된장 또는 미소(Miso)를 추가하여 깊은 한국식 발효 풍미를 더합니다. |

---

### 비건 김치찌개 조리 가이드

1.  **재료 준비:** 식물성 고기(예: 콩고기, 버섯류 등)를 먹기 좋은 크기로 자릅니다.
2.  **볶기:** 냄비에 식물성 고기와 김치를 넣고 다진 마늘을 넣어 함께 볶아줍니다. (이 과정에서 재료의 풍미가 깊어집니다.)
3.  **끓이기:** 채소 육수(또는 물)와 준비된 감칠맛 재료(선택 사항: 된장/미소)를 넣고 센 불로 끓입니다.
4.  **마무리:** 끓기 시작하면 중약불로 줄이고, 두부를 넣은 후 맛을 보며 충분히 끓여줍니다. 마지막에 대파를 넣어 향을 더합니다.

### 재료 대체 상세 설명

*   **돼지고기 (200g) → 식물성 고기:** 육류 대신 콩고기나 버섯 등을 사용하여 단백질과 식감을 보충합니다.
*   **멸치육수 (2컵) → 채소 육수:** 해산물 기반의 감칠맛 대신 무, 양파 등 채소를 베이스로 하여 비건 친화적인 깊은 맛을 냅니다.


### 8.3 고객 후기 자동 분류

쇼핑몰 또는 앱스토어 후기 분석 파이프라인에 그대로 적용 가능한 패턴입니다. 평점, 카테고리, 키워드, 응대 톤을 한 번의 호출로 추출합니다.

In [11]:
reviews = [
    "배송이 너무 느렸어요. 일주일이나 걸렸네요. 제품은 괜찮은데 다음엔 다른 곳에서 살 듯",
    "디자인 진짜 예뻐요!! 친구한테 선물했는데 너무 좋아하네요 ㅎㅎ 가격도 합리적",
    "사이즈가 너무 작아요. 표기는 L인데 실제로는 M 같은 느낌. 교환 가능한가요?",
]

for review in reviews:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "쇼핑몰 후기를 분석해 JSON으로 응답: "
                                           '{"score": 1~5(int), "category": "배송|품질|디자인|사이즈|가격|기타", '
                                           '"keywords": [3개 이내], "reply_tone": "사과|감사|안내"}'},
            {"role": "user", "content": review},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    print(f"📝 {review[:30]}...")
    print(f"   → {r.choices[0].message.content}\n")

📝 배송이 너무 느렸어요. 일주일이나 걸렸네요. 제품은 괜...
   → {
"score": 3,
"category": "배송",
"keywords": [
"배송 지연",
"느림",
"제품 만족"],
"reply_tone": "사과"
}

📝 디자인 진짜 예뻐요!! 친구한테 선물했는데 너무 좋아하...
   → {
  "score": 5,
  "category": "디자인",
  "keywords": [
    "디자인",
    "가격",
    "선물"
  ],
  "reply_tone": "감사"
}

📝 사이즈가 너무 작아요. 표기는 L인데 실제로는 M 같은...
   → {
"score": 2,
"category": "사이즈",
"keywords": [
"사이즈",
"작음",
"교환"
],
"reply_tone": "사과"
}



---

## 실습 과제: 미니 챗봇 구현

다음 요구사항을 만족하는 간단한 챗봇을 직접 구현해 보세요.

**요구사항**

1. `system` 프롬프트로 챗봇의 역할(예: 사내 IT 지원 도우미)을 명확히 설정합니다.
2. `temperature=0.3`으로 일관된 응답을 유도합니다.
3. 사용자가 `quit`을 입력하면 루프를 종료합니다.
4. 각 응답 출력 시 사용된 토큰 수를 함께 표시합니다.

In [12]:
# 여기에 코드를 작성하세요
# Hint: while 루프 + history 리스트 + response.usage.total_tokens 활용


<details>
<summary>모범 답안 보기</summary>

```python
# 노트북 상단에서 초기화한 client / MODEL 변수를 그대로 사용합니다 (Gemini 또는 OpenAI 자동 분기).
history = [{"role": "system", "content": "당신은 사내 IT 지원 도우미입니다. 간결하고 정확하게 답하세요."}]

while True:
    user_input = input("당신: ")
    if user_input.strip().lower() == "quit":
        print("종료합니다.")
        break
    history.append({"role": "user", "content": user_input})
    r = client.chat.completions.create(
        model=MODEL,
        messages=history,
        temperature=0.3,
    )
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    print(f"봇: {reply}  (토큰: {r.usage.total_tokens})")
```

</details>

---

## 트러블슈팅 — 자주 마주치는 오류

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `RateLimitError: 429` | 분당 요청 한도 초과 | 30~60초 대기 후 재시도. Gemini 사용 한도는 분당 15회입니다. |
| `AuthenticationError: 401` | API 키 오타 또는 만료 | `.env` 파일 재확인. 키 앞뒤 공백 여부 점검. |
| `BadRequestError: model not found` | 모델 식별자 오타 | Gemini는 `gemini-2.5-flash-lite`, OpenAI는 `gpt-4.1-mini`로 지정. |
| `json.JSONDecodeError` | 모델이 JSON 스펙을 준수하지 않음 | `temperature=0` 사용, 시스템 프롬프트에서 출력 형식을 명시. |
| 한국어 출력 품질 저하 | 일부 모델의 한국어 능력 한계 | Gemini 또는 Qwen3 사용 권장. (Llama 3.3은 한국어 다소 약함) |

```python
# 예외 처리를 포함한 안전한 호출 예시
try:
    r = client.chat.completions.create(model=MODEL, messages=[...])
except Exception as e:
    print(f"LLM 호출 실패: {type(e).__name__}: {e}")
    # 재시도, 폴백 모델로 전환, 또는 사용자 안내 등의 후속 처리를 수행합니다.
```

---

## 마무리

본 노트북에서 학습한 내용은 다음과 같습니다.

- LLM의 동작 원리(확률 기반 다음 토큰 예측)
- OpenAI SDK를 사용한 기본 호출 방법(`client.chat.completions.create`)
- `temperature`를 통한 출력 다양성 제어
- `response_format`을 활용한 JSON 구조화 출력
- 메시지 이력 누적을 통한 멀티턴 대화 구현
- 한국어 비즈니스 시나리오에 대한 적용 사례

### 다음 노트북(02번) 예고

본 노트북의 코드는 OpenAI SDK 인터페이스에 결합되어 있어, Claude나 Gemini의 네이티브 SDK로 전환할 경우 호출부를 모두 다시 작성해야 합니다.

**LangChain의 핵심 가치**는 바로 이 지점에 있습니다. `init_chat_model("gpt-4.1-mini")`와 `init_chat_model("gemini-2.5-flash-lite")`는 동일한 인터페이스를 제공하므로, 모델 교체에 따른 코드 수정을 최소화할 수 있습니다.

다음 노트북에서는 LangChain v1.0의 **LCEL(LangChain Expression Language)** 을 사용해 "프롬프트 → 모델 → 파서"를 파이프라인 한 줄로 연결하는 방법을 다룹니다.